# Bedrock LLM Gateway — Quick Test

Minimal notebook that calls the AWS LLM Gateway (Ollama-compatible API fronting Claude Sonnet 4.5 on Bedrock) and prints the reply.

- Stdlib only — no `pip install` needed.
- Reads `LLM_GATEWAY_URL` / `LLM_GATEWAY_API_KEY` / `LLM_MODEL` from `.env.local` in this same folder (the file `src/lib/ai/gateway.ts` also uses), or from real environment variables if already set.
- Same `/api/chat` protocol as `gateway.ts` and the Starter Kit's `weather_demo.py` — just a plain HTTP POST with an `X-API-Key` header.

In [1]:
import json
import os
import urllib.error
import urllib.request


def load_dotenv(path=".env.local"):
    """Minimal .env loader — real env vars take precedence."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, _, value = line.partition("=")
                os.environ.setdefault(key.strip(), value.strip())
    except FileNotFoundError:
        pass


load_dotenv()

LLM_GATEWAY_URL = os.getenv("LLM_GATEWAY_URL")
LLM_GATEWAY_API_KEY = os.getenv("LLM_GATEWAY_API_KEY")
LLM_MODEL = os.getenv("LLM_MODEL")

assert all([LLM_GATEWAY_URL, LLM_GATEWAY_API_KEY, LLM_MODEL]), (
    "Missing LLM_GATEWAY_URL / LLM_GATEWAY_API_KEY / LLM_MODEL — "
    "make sure .env.local sits next to this notebook (run this notebook "
    "from the design-inspiration-agent folder)."
)
print(f"Gateway: {LLM_GATEWAY_URL}")
print(f"Model:   {LLM_MODEL}")

Gateway: https://api.softwaresystems.app
Model:   global.anthropic.claude-sonnet-4-5-20250929-v1:0


In [2]:
def chat(messages, num_predict=200):
    """POST /api/chat to the gateway (non-streaming). Returns the parsed JSON response."""
    payload = {
        "model": LLM_MODEL,
        "messages": messages,
        "stream": False,
        "options": {"num_predict": num_predict},
    }
    req = urllib.request.Request(
        LLM_GATEWAY_URL.rstrip("/") + "/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "X-API-Key": LLM_GATEWAY_API_KEY,
        },
        method="POST",
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            return json.loads(resp.read())
    except urllib.error.HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Gateway error HTTP {e.code}: {body}") from e

In [3]:
# Test 1: simple one-shot call
response = chat([
    {"role": "user", "content": "Say hello in exactly one short sentence."},
])
print(response["message"]["content"])

Hello!


In [4]:
# Test 2: your own question, with the raw response printed too (handy for debugging shape/fields)
response = chat([
    {"role": "user", "content": "how many toes do donkeys have in total"},
])
print("Reply:\n", response["message"]["content"])
print("\nRaw response:\n", json.dumps(response, indent=2))

Reply:
 Donkeys have **4 toes in total** - one hoof on each of their four feet.

Like horses, donkeys are "odd-toed ungulates" (perissodactyls), but the "odd-toed" classification refers to the evolutionary group rather than the actual number of toes visible. Modern donkeys actually walk on a single toe (the third toe) on each foot, which has evolved into a hoof. So while they have one functional toe per foot (4 total), they also have small remnant bones called splint bones on either side of each leg that represent reduced second and fourth toes from their evolutionary ancestors.

Raw response:
 {
  "model": "global.anthropic.claude-sonnet-4-5-20250929-v1:0",
  "created_at": "2026-09-11T15:28:22.392076Z",
  "message": {
    "role": "assistant",
    "content": "Donkeys have **4 toes in total** - one hoof on each of their four feet.\n\nLike horses, donkeys are \"odd-toed ungulates\" (perissodactyls), but the \"odd-toed\" classification refers to the evolutionary group rather than the actu